In [ ]:

# %pip install -U langchain langchain-ollama langgraph langsmith llama-index-core llama-index-llms-ollama llama-index-embeddings-ollama

In [ ]:
from dotenv import load_dotenv

from langchain_core.tools import tool

from config import PARQUET_CLEAN_DATASET

import duckdb

NUTRI_COLS = {  # noms bruts, sans guillemets SQL
    "kcal": "energy-kcal_100g",
    "lipides": "fat_100g",
    "satures": "saturated-fat_100g",
    "sucres": "sugars_100g",
    "sel": "salt_100g",
    "fibres": "fiber_100g",
    "proteines": "proteins_100g",
}


def _build_filter(nom: str | None, code: str | None) -> str:
    if code:
        return f"code = {code}"
    return f"strip_accents(product_name) ILIKE '%' || strip_accents('{nom}') || '%'"

@tool(parse_docstring=True)
def find_product(name: str | None = None, code: str | None = None, limit: int = 3) -> dict:
    """
    Cherche un produit dans le catalogue NutriScope et renvoie ses fiches :
        Nutri-Score (A à E, ou null si inconnu) et nutriments pour 100 g (null si inconnu).
        Seule source autorisée pour toute information sur un produit.
        Donner nom ou code (au moins l'un des deux). Aucun autre argument.


    Args:
        name: Nom ou partie du nom du produit, sans les mots de la question
        code: Code-barres exact, en texte. Prime sur nom si les deux sont donnés.
        limit: Nombre maximal de fiches renvoyées : entier de 1 à 5. Facultatif, 3 par défaut.
    """
    res = {
        "status": "introuvable",
        "total": 0,
        "products": []
    }
    if name is None and code is None and 0 < limit < 6:
        return res
    where_statement = _build_filter(name, code)

    struct_fields = ", ".join(
        f'{k} := CASE WHEN "{col}_empty" THEN NULL ELSE "{col}" END'
        for k, col in NUTRI_COLS.items()
    )
    any_missing = " OR ".join(f'"{col}_empty"' for col in NUTRI_COLS.values())

    con = duckdb.connect()
    response = con.execute(
        f"""
        SELECT code::VARCHAR as code,
        product_name as name,
        CASE WHEN
            lower(nutriscore_grade) IN ('a','b','c','d','e')
            THEN upper(nutriscore_grade) END AS nutriscore ,
            struct_pack({struct_fields}) AS nutrients,
            (nutriscore IS NULL OR {any_missing}) AS incomplete_data
        FROM read_parquet('{PARQUET_CLEAN_DATASET}')
        where {where_statement}
        limit {limit}
        """
    ).df()
    products = []
    total = len(response)
    res["total"] = total
    if total == 0:
        return res
    elif total == 1:
        res["status"] = "ok"
    else:
        res["status"] = "ambiguous"
    for row in response.values:
        products.append({
            "code": row[0],
            "name": row[1],
            "nutriscore": row[2],
            "nutrients": row[3],
            "incomplete_data": row[4]
        })
    res["products"] = products

    return res

In [38]:
import time
from langchain_core.callbacks import BaseCallbackHandler


class Journal(BaseCallbackHandler):
    """Note chaque étape de l'agent : type, nom, durée et tokens. Jamais le texte des messages."""

    def __init__(self):
        self.lignes = []        # une ligne par étape terminée
        self.debuts = {}        # l'heure de début de chaque étape en cours, rangée par numéro d'étape

    # Un appel au modèle commence : on note l'heure
    def on_chat_model_start(self, serialized, messages, *, run_id, **kwargs):
        self.debuts[run_id] = time.perf_counter()

    # L'appel au modèle se termine : on écrit sa ligne
    def on_llm_end(self, response, *, run_id, **kwargs):
        duree = time.perf_counter() - self.debuts[run_id]
        message = response.generations[0][0].message        # le message rendu par le modèle

        # Le nom de l'étape : les outils demandés par le modèle, sinon « réponse »
        outils_demandes = []
        for appel in message.tool_calls:
            outils_demandes.append(appel["name"])
        if outils_demandes:
            nom = ", ".join(outils_demandes)
        else:
            nom = "réponse"

        tokens = message.usage_metadata
        ligne = {"étape": "modèle", "nom": nom, "durée (s)": round(duree, 2),
                 "tokens entrée": tokens["input_tokens"], "tokens sortie": tokens["output_tokens"]}
        self.lignes.append(ligne)

    # Un outil commence : on note l'heure
    def on_tool_start(self, serialized, input_str, *, run_id, **kwargs):
        self.debuts[run_id] = time.perf_counter()

    # L'outil se termine : son résultat (output) porte le nom de l'outil
    def on_tool_end(self, output, *, run_id, **kwargs):
        duree = time.perf_counter() - self.debuts[run_id]
        ligne = {"étape": "outil", "nom": output.name, "durée (s)": round(duree, 2),
                 "tokens entrée": 0, "tokens sortie": 0}
        self.lignes.append(ligne)

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
URL = os.getenv("LLM_URL")
MODEL = os.getenv("LLM_MODEL")
EMB = os.getenv("MODEL_EMBEDDING")

print(URL, MODEL, EMB)

In [34]:
from langchain_ollama import ChatOllama

def fabriquer_modele(temperature=0, num_predict=5000):
    if (os.getenv("LLM_PROVIDER") or "ollama") == "ollama":
        return ChatOllama(
            model=os.getenv("OLLAMA_MODEL", "qwen3:8b"),
            base_url=os.getenv("OLLAMA_URL", "http://localhost:11434"),
            temperature=temperature, # 0 pour des réponses déterministes/factuelles
            reasoning=False,         # Désactive la réflexion préalable si non nécessaire
            num_predict=num_predict  # Plafond de tokens générés
        )
    from langchain_openai import ChatOpenAI
    return ChatOpenAI(
        model=os.environ["LLM_MODEL"],
        base_url=os.environ["LLM_BASE_URL"],
        api_key=os.environ["LLM_API_KEY"],
        temperature=temperature,
        max_tokens=num_predict
    )

modele = fabriquer_modele()

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from llm.config_assistant import PROMPT_SYSTEME
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", PROMPT_SYSTEME),
    ("human", "{message}")
])

first_chain = prompt | modele | StrOutputParser()

resp = first_chain.invoke({"message": "Donne moi les infos de la pate à tartiner nutella"})

In [53]:
import pandas as pd
from langchain_core.prompts import MessagesPlaceholder
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

journal = Journal()

agent = create_agent(model=modele, tools=[find_product], system_prompt=PROMPT_SYSTEME)

prompt_conv = ChatPromptTemplate.from_messages([
    MessagesPlaceholder("historique"),
    ("human", "{message}")
])

chain_conv = prompt | agent

def call_llm(hist, message):
    return chain_conv.invoke({"historique": hist, "message": message}, config={"callbacks": [journal]})["messages"]

def talk_to_llm() -> str:
    hist = []
    while True:
        message = input(f"\n Vous > ").strip()
        print(message)
        llm_response = call_llm(hist, message)
        hist.extend([HumanMessage(message), llm_response])
        print(f"\n {MODEL} >",llm_response, end=" ", )
        print(hist)

In [54]:
talk_to_llm()

Bonjour

 qwen3:8b > [SystemMessage(content="Tu es l'assistant conversationnel de NutriScope, une application d'aide alimentaire.", additional_kwargs={}, response_metadata={}, id='b41b66a7-b83e-4426-a732-7b8825500333'), HumanMessage(content='Bonjour', additional_kwargs={}, response_metadata={}, id='99f863a1-b66b-467e-b4e5-ace440118627'), AIMessage(content="Bonjour ! Je suis l'assistant NutriScope. Je peux vous aider à trouver des informations sur les produits alimentaires du catalogue : Nutri-Score, valeurs nutritionnelles, comparaisons, explications des scores et alternatives mieux notées. N'hésitez pas à me poser vos questions !", additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-11-23T15:35:21.0576689Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1260069900, 'load_duration': 4675500, 'prompt_eval_count': 2053, 'prompt_eval_duration': 82409000, 'eval_count': 64, 'eval_duration': 1155067000, 'logprobs': None, 'model_name': 'qwen3:8b', 'model

KeyboardInterrupt: Interrupted by user